# Per-Layer Adaptive Fusion v2 — CMU-MOSEI Demo

This notebook demonstrates **`PLGModelV2`** (`fusions/per_layer_gated_v2.py`), which reimplements
the per-layer gating design from `affect_dyn_v2.py` with two structural changes over v1
(`fusions/per_layer_gated.py`).

## v1 vs v2 differences

| Aspect | v1 (`per_layer_gated.py`) | v2 (`per_layer_gated_v2.py`) |
|---|---|---|
| Gate source | Local: `Linear(d_model→1)` on each modality's own hidden state, computed inline as encoding proceeds | **Centralized**: single `PerLayerGate` sees all raw inputs concatenated, outputs full `[B, M, L]` matrix **upfront** |
| Gate mechanism | Gated residual: `g·h_new + (1-g)·h_prev` (always blends previous state back in) | **Multiplicative zeroing**: `h = h * g` — gate=0 fully suppresses that layer's output |
| Hard gating | Not supported | **Straight-through sigmoid**: `{0,1}` forward pass, soft gradient backward |
| Resource loss | `gates.mean()` (uniform penalty) | **FLOPs-weighted**: `Σ(gates_mean × flop_cost_per_layer)` |
| Infer modes | None | `0`=learned · `1`=text-only · `-1`=all-on |
| Pretrained init | Not supported | `init_from_pretrained()` |

### Dataset
CMU-MOSEI sentiment regression · visual=35d · audio=74d · text=300d GloVe

## 1 · Setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.isdir('/content/MultiBench'):
        !git clone https://github.com/pliang279/MultiBench /content/MultiBench
    REPO_ROOT = '/content/MultiBench'
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('Repository root:', REPO_ROOT)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/plg_v2_checkpoints'
else:
    CHECKPOINT_DIR = os.path.join(REPO_ROOT, 'log', 'mosei')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoint dir:', CHECKPOINT_DIR)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

try:
    from scipy.stats import pearsonr
except ImportError:
    !pip install -q scipy
    from scipy.stats import pearsonr

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## 2 · Data

CMU-MOSEI pickle: Google Drive folder `1A_hTmifi824gypelGobgl2M-5Rw9VWHv`

In [ ]:
import types

# Mock torchtext — not needed for preprocessed MOSEI data
fake_tt = types.ModuleType('torchtext')
fake_tt.vocab = types.ModuleType('torchtext.vocab')
sys.modules.setdefault('torchtext', fake_tt)
sys.modules.setdefault('torchtext.vocab', fake_tt.vocab)

if IN_COLAB:
    DATA_PATH = '/content/drive/MyDrive/mosei/mosei_senti_data.pkl'
    if not os.path.exists(DATA_PATH):
        !pip install -q gdown
        import gdown
        gdown.download_folder(
            'https://drive.google.com/drive/folders/1A_hTmifi824gypelGobgl2M-5Rw9VWHv',
            output='/content/mosei_data', quiet=False)
        DATA_PATH = '/content/mosei_data/mosei_senti_data.pkl'
else:
    DATA_PATH = os.path.join(REPO_ROOT, 'data', 'mosei_senti_data.pkl')

assert os.path.exists(DATA_PATH), f'Not found: {DATA_PATH}'
print('Data path:', DATA_PATH)

from datasets.affect.get_data import get_dataloader

traindata, validdata, testdata = get_dataloader(
    DATA_PATH, batch_size=32,
    max_pad=True, max_seq_len=50,
    robust_test=False, data_type='mosei', num_workers=0,
)

v, a, t, y = next(iter(traindata))
print(f'visual: {tuple(v.shape)}  audio: {tuple(a.shape)}  text: {tuple(t.shape)}  labels: {tuple(y.shape)}')
print(f'train={len(traindata)}  val={len(validdata)}  test={len(testdata)} batches')

## 3 · Architecture

```
PLGModelV2
├── PerLayerGatedFusionV2
│   ├── PerLayerGate          ← centralized: sees concat(v,a,t) [B,T,409]
│   │   ├── Conv1d(409→10)
│   │   ├── TransformerEncoderLayer(d=10, nhead=1)
│   │   └── Linear(10 → 3×5=15)  →  gates [B, 3, 5]
│   ├── GatedTransformerEncoder  visual  35→60d, 5 layers, nhead=5
│   ├── GatedTransformerEncoder  audio   74→120d
│   └── GatedTransformerEncoder  text   300→120d
└── Linear(300→1)  ← regression head
```

The gate network decides all 15 routing decisions **before** any encoding begins,
allowing cross-modal context (e.g. "audio SNR is low — suppress audio at all layers").
Each encoder then applies its gate slice via multiplicative zeroing:
```
h_l = TransformerLayer_l(h_{l-1})
h_l = h_l * g_l          # gate=0 → fully suppressed; gate=1 → full pass-through
```

In [ ]:
from fusions.per_layer_gated_v2 import PLGModelV2, GatedTransformerEncoder
from fusions.per_layer_gated import PLGModel as PLGModelV1

# Hyperparameters matching affect_dyn_v2.py
IN_DIMS    = [35, 74, 300]
D_MODELS   = [60, 120, 120]   # per-modality encoder dims
NUM_LAYERS = 5
NHEAD      = 5
HIDDEN_DIM = 10               # gate network internal size

def make_v2(hard_gate=False, temp=1.0):
    return PLGModelV2(
        in_dims=IN_DIMS, d_models=D_MODELS,
        num_layers=NUM_LAYERS, nhead=NHEAD,
        hidden_dim=HIDDEN_DIM, temp=temp,
        out_dim=1, hard_gate=hard_gate,
    )

probe = make_v2()
n_params_v2 = sum(p.numel() for p in probe.parameters() if p.requires_grad)
print(f'PLGModelV2  params: {n_params_v2:,}')
print(probe)

In [ ]:
# Quick shape check — no GPU required
with torch.no_grad():
    dummy = [torch.randn(4, 50, d) for d in IN_DIMS]
    pred, reg = probe(dummy)
    fused, gates = probe.fusion(dummy)

print(f'pred   : {tuple(pred.shape)}    expected (4, 1)')
print(f'fused  : {tuple(fused.shape)}  expected (4, 300)  [sum of d_models]')
print(f'gates  : {tuple(gates.shape)}  expected (4, 3, 5)  [B, modalities, layers]')
print(f'reg    : {reg.item():.4f}    FLOPs-weighted gate penalty at init')
print(f'gate range at init: [{gates.min():.3f}, {gates.max():.3f}]  (expect ~0.5)')

## 4 · Baselines

We train **five** reference models spanning the unimodal → multimodal spectrum from Table 2 of the DynMM paper:

| Model | Description |
|---|---|
| **Video-only** | Visual encoder alone. Matches 'Video Network' in DynMM Table 2. |
| **Audio-only** | Audio encoder alone. Matches 'Audio Network' in DynMM Table 2. |
| **Text-only** | Text encoder alone. Matches DynMM's E1 branch. |
| **Early Fusion** | Concat all inputs feature-wise, single encoder. Matches 'Early Fusion [24]' in Table 2. |
| **Late Fusion** | Separate encoders, concat representations. Matches DynMM's E2 branch. |
| **PLGModel v1** | Per-layer gated residuals, local gates (`per_layer_gated.py`). Direct v1→v2 comparison. |

Full DynMM paper reference (Table 2, CMU-MOSEI):

| Method | MAE | Acc2 | MAdds |
|---|---|---|---|
| Video Network | 0.80 | 69.02% | 123.1M |
| Audio Network | 0.82 | 67.68% | 123.3M |
| Text Network E1 | 0.62 | 78.35% | 124.7M |
| Early Fusion | 0.65 | 78.45% | 313.5M |
| Late Fusion E2 | 0.60 | 79.54% | 309.6M |
| DynMM-a | 0.62 | 79.07% | 165.5M |
| DynMM-b | 0.61 | 79.73% | 254.5M |
| DynMM-c | 0.60 | 79.75% | 295.8M |

In [ ]:
class LateFusionBaseline(nn.Module):
    """Three GatedTransformerEncoders with no gate network (always all layers active).

    Matches DynMM's E2 branch (Late Fusion [24], Table 2 of DynMM paper).
    """
    def __init__(self):
        super().__init__()
        self.encoders = nn.ModuleList([
            GatedTransformerEncoder(IN_DIMS[m], D_MODELS[m], num_layers=NUM_LAYERS, nhead=NHEAD)
            for m in range(3)
        ])
        self.head = nn.Linear(sum(D_MODELS), 1)

    def forward(self, inputs):
        inputs = [t.float() for t in inputs]
        reps = [enc(x) for enc, x in zip(self.encoders, inputs)]
        return self.head(torch.cat(reps, dim=-1)), torch.tensor(0.0, device=inputs[0].device)


class TextOnlyBaseline(nn.Module):
    """Single text transformer encoder + head.

    Matches DynMM's E1 branch (Text Network, Table 2 of DynMM paper).
    """
    def __init__(self):
        super().__init__()
        self.encoder = GatedTransformerEncoder(300, 120, num_layers=NUM_LAYERS, nhead=NHEAD)
        self.head = nn.Linear(120, 1)

    def forward(self, inputs):
        text = inputs[-1].float()  # last modality is text
        return self.head(self.encoder(text)), torch.tensor(0.0, device=text.device)


class VideoOnlyBaseline(nn.Module):
    """Single visual transformer encoder + head.

    Matches 'Video Network' (Table 2 of DynMM paper).
    """
    def __init__(self):
        super().__init__()
        self.encoder = GatedTransformerEncoder(IN_DIMS[0], D_MODELS[0], num_layers=NUM_LAYERS, nhead=NHEAD)
        self.head = nn.Linear(D_MODELS[0], 1)

    def forward(self, inputs):
        video = inputs[0].float()
        return self.head(self.encoder(video)), torch.tensor(0.0, device=video.device)


class AudioOnlyBaseline(nn.Module):
    """Single audio transformer encoder + head.

    Matches 'Audio Network' (Table 2 of DynMM paper).
    """
    def __init__(self):
        super().__init__()
        self.encoder = GatedTransformerEncoder(IN_DIMS[1], D_MODELS[1], num_layers=NUM_LAYERS, nhead=NHEAD)
        self.head = nn.Linear(D_MODELS[1], 1)

    def forward(self, inputs):
        audio = inputs[1].float()
        return self.head(self.encoder(audio)), torch.tensor(0.0, device=audio.device)


class EarlyFusionBaseline(nn.Module):
    """Concatenate all modalities at the input level, process with a single encoder.

    Matches 'Early Fusion [24]' (Table 2 of DynMM paper).
    All three modality sequences are trimmed to the shortest length and
    concatenated along the feature dimension (35+74+300=409) before encoding.
    """
    def __init__(self):
        super().__init__()
        total_dim = sum(IN_DIMS)   # 35+74+300 = 409
        d_out = sum(D_MODELS)      # 60+120+120 = 300
        self.encoder = GatedTransformerEncoder(total_dim, d_out, num_layers=NUM_LAYERS, nhead=NHEAD)
        self.head = nn.Linear(d_out, 1)

    def forward(self, inputs):
        inputs = [t.float() for t in inputs]
        min_t = min(x.shape[1] for x in inputs)
        fused = torch.cat([x[:, :min_t, :] for x in inputs], dim=-1)  # (B, T, 409)
        return self.head(self.encoder(fused)), torch.tensor(0.0, device=fused.device)



class DynMMBaseline(nn.Module):
    """Modality-level DynMM: a gate selects per-sample between two expert branches.

    E1 = text-only  (same arch as TextOnlyBaseline)  ← DynMM's cheap branch
    E2 = late fusion (same arch as LateFusionBaseline) ← DynMM's rich branch
    Gate = 1-layer GatedTransformerEncoder on concat(v,a,t) -> 2-way Gumbel-softmax

    This is architecturally equivalent to DynMM (Table 2 of the DynMM paper) but
    uses GatedTransformerEncoder throughout, so it works with this notebook's plain-
    tensor data loader and shares the same evaluate() call as every other model here.

    Two training modes:
      - End-to-end: all weights trained from scratch together
      - Two-stage:  call init_from_baselines(txt, lf) to seed E1/E2 from already-
                    trained TextOnlyBaseline / LateFusionBaseline (DynMM Stage I/II)
    """
    # Approximate MAdds (M) for each expert branch
    # 4 * d^2 * layers per encoder (attn + FFN estimate)
    _E1_MADDS = 4 * 120**2 * 5 / 1e6          # text encoder
    _E2_MADDS = 4 * sum(d**2 for d in [60, 120, 120]) * 5 / 1e6  # 3 encoders

    def __init__(self, hard_gate=False, temp=1.0):
        super().__init__()
        # E1: text-only expert
        self.e1_encoder = GatedTransformerEncoder(300, 120, num_layers=NUM_LAYERS, nhead=NHEAD)
        self.e1_head    = nn.Linear(120, 1)
        # E2: late-fusion expert
        self.e2_encoders = nn.ModuleList([
            GatedTransformerEncoder(IN_DIMS[m], D_MODELS[m], num_layers=NUM_LAYERS, nhead=NHEAD)
            for m in range(3)
        ])
        self.e2_head = nn.Linear(sum(D_MODELS), 1)
        # Gate: lightweight 1-layer transformer on concat(v,a,t) -> 2 logits
        self.gate_enc = GatedTransformerEncoder(sum(IN_DIMS), HIDDEN_DIM, num_layers=1, nhead=1)
        self.gate_lin = nn.Linear(HIDDEN_DIM, 2)
        self.hard_gate = hard_gate
        self.temp      = temp
        self._gate_log = []  # accumulates (B,2) gate weights during eval

    def init_from_baselines(self, txt_model, lf_model):
        """Seed E1/E2 from pre-trained baselines (DynMM Stage I init)."""
        self.e1_encoder.load_state_dict(txt_model.encoder.state_dict())
        self.e1_head.load_state_dict(txt_model.head.state_dict())
        self.e2_encoders.load_state_dict(lf_model.encoders.state_dict())
        self.e2_head.load_state_dict(lf_model.head.state_dict())

    def reset_weight(self):
        self._gate_log = []

    def cal_flop(self):
        """Effective MAdds = mean_g1 * E1_madds + mean_g2 * E2_madds."""
        if not self._gate_log:
            return 0.0
        mean_g = torch.cat(self._gate_log).mean(0)  # shape (2,)
        return float(mean_g[0] * self._E1_MADDS + mean_g[1] * self._E2_MADDS)

    def forward(self, inputs):
        inputs = [x.float() for x in inputs]
        min_t  = min(x.shape[1] for x in inputs)
        concat = torch.cat([x[:, :min_t, :] for x in inputs], dim=-1)  # (B,T,409)
        logits = self.gate_lin(self.gate_enc(concat))                   # (B,2)
        if self.training:
            g = torch.nn.functional.gumbel_softmax(logits, tau=self.temp, hard=self.hard_gate)
        else:
            if self.hard_gate:
                idx = logits.argmax(-1, keepdim=True)
                g = torch.zeros_like(logits).scatter_(-1, idx, 1.0)
            else:
                g = torch.nn.functional.softmax(logits / self.temp, dim=-1)
            self._gate_log.append(g.detach().cpu())
        # Expert predictions
        p1 = self.e1_head(self.e1_encoder(inputs[2]))                          # (B,1)
        p2 = self.e2_head(torch.cat(
            [enc(x) for enc, x in zip(self.e2_encoders, inputs)], dim=-1))     # (B,1)
        # Weighted combination + resource loss (fraction routing to heavy E2)
        output = g[:, 0:1] * p1 + g[:, 1:2] * p2
        reg    = g[:, 1].mean()
        return output, reg

v1_probe = PLGModelV1(in_dims=IN_DIMS, d_model=64, n_heads=4, n_layers=NUM_LAYERS)
print(f'LateFusionBaseline  params: {sum(p.numel() for p in LateFusionBaseline().parameters()):,}')
print(f'TextOnlyBaseline    params: {sum(p.numel() for p in TextOnlyBaseline().parameters()):,}')
print(f'VideoOnlyBaseline   params: {sum(p.numel() for p in VideoOnlyBaseline().parameters()):,}')
print(f'AudioOnlyBaseline   params: {sum(p.numel() for p in AudioOnlyBaseline().parameters()):,}')
print(f'EarlyFusionBaseline params: {sum(p.numel() for p in EarlyFusionBaseline().parameters()):,}')
print(f'DynMMBaseline       params: {sum(p.numel() for p in DynMMBaseline().parameters()):,}')
print(f'PLGModel v1         params: {sum(p.numel() for p in v1_probe.parameters()):,}')
print(f'PLGModel v2         params: {n_params_v2:,}')

## 5 · Training

All models are trained end-to-end from scratch on CMU-MOSEI.

**Total loss**: `L = MAE(ŷ, y) + λ · resource_loss`
- Baselines: `resource_loss = 0`
- PLGModelV2: `resource_loss = Σ(mean_gates × flop_cost_per_layer)` (FLOPs-weighted)
- PLGModel v1: `resource_loss = mean(gates)` (uniform)

> Set `EPOCHS = 40` for paper-comparable results. Default `10` for a quick demo.

In [ ]:
EPOCHS = 10   # ← increase to 40 for full training
LR     = 1e-4
WD     = 1e-4


def train_model(model, name, lossw=0.0):
    """Generic training loop for any model returning (pred, reg_loss)."""
    save_path = os.path.join(CHECKPOINT_DIR, f'{name}.pt')
    model.to(DEVICE)
    opt  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    crit = nn.L1Loss()
    best_val, pat = float('inf'), 0
    print(f'\n── {name}  (λ={lossw}, {EPOCHS} epochs) ──')

    for epoch in range(EPOCHS):
        model.train()
        tl = 0.0
        for batch in traindata:
            v, a, t, y = batch
            inputs = [v.float().to(DEVICE), a.float().to(DEVICE), t.float().to(DEVICE)]
            opt.zero_grad()
            pred, reg = model(inputs)
            loss = crit(pred, y.float().to(DEVICE)) + lossw * reg
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 8.0)
            opt.step()
            tl += loss.item()

        model.eval()
        vl = 0.0
        with torch.no_grad():
            for batch in validdata:
                v, a, t, y = batch
                inputs = [v.float().to(DEVICE), a.float().to(DEVICE), t.float().to(DEVICE)]
                pred, reg = model(inputs)
                vl += (crit(pred, y.float().to(DEVICE)) + lossw * reg).item()

        avg_t, avg_v = tl / len(traindata), vl / len(validdata)
        mark = ''
        if avg_v < best_val:
            best_val = avg_v
            torch.save(model.state_dict(), save_path)
            mark = '  ← best'
            pat = 0
        else:
            pat += 1
        print(f'  Epoch {epoch+1:>3}/{EPOCHS} | train {avg_t:.4f} | val {avg_v:.4f}{mark}')
        if pat >= 8:
            print(f'  Early stop at epoch {epoch+1}')
            break

    return save_path


def evaluate(model):
    """Return dict with MAE, Corr, Acc (binary pos/neg classification)."""
    preds, labs = [], []
    model.eval()
    with torch.no_grad():
        for batch in testdata:
            v, a, t, y = batch
            inputs = [v.float().to(DEVICE), a.float().to(DEVICE), t.float().to(DEVICE)]
            p, _ = model(inputs)
            preds.append(p.squeeze(-1).cpu())
            labs.append(y.squeeze(-1).cpu())
    p = torch.cat(preds).numpy()
    l = torch.cat(labs).numpy()
    corr, _ = pearsonr(p, l)
    return {'MAE': float(np.abs(p - l).mean()),
            'Corr': float(corr),
            'Acc':  float(np.mean((p >= 0) == (l >= 0)))}

In [ ]:
results = {}  # label -> test metrics dict
trained_models = {}

# -- Late Fusion Baseline -------------------------------------------------
lf = LateFusionBaseline()
path = train_model(lf, 'lf_baseline', lossw=0.0)
lf.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
lf.to(DEVICE)
results['Late Fusion (baseline)'] = evaluate(lf)
trained_models['Late Fusion (baseline)'] = lf
print('  ->', results['Late Fusion (baseline)'])

# -- Text-Only Baseline ---------------------------------------------------
txt = TextOnlyBaseline()
path = train_model(txt, 'txt_baseline', lossw=0.0)
txt.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
txt.to(DEVICE)
results['Text-only (baseline)'] = evaluate(txt)
trained_models['Text-only (baseline)'] = txt
print('  ->', results['Text-only (baseline)'])

# -- Video-Only Baseline --------------------------------------------------
vid = VideoOnlyBaseline()
path = train_model(vid, 'vid_baseline', lossw=0.0)
vid.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
vid.to(DEVICE)
results['Video-only (baseline)'] = evaluate(vid)
trained_models['Video-only (baseline)'] = vid
print('  ->', results['Video-only (baseline)'])

# -- Audio-Only Baseline --------------------------------------------------
aud = AudioOnlyBaseline()
path = train_model(aud, 'aud_baseline', lossw=0.0)
aud.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
aud.to(DEVICE)
results['Audio-only (baseline)'] = evaluate(aud)
trained_models['Audio-only (baseline)'] = aud
print('  ->', results['Audio-only (baseline)'])

# -- Early Fusion Baseline ------------------------------------------------
ef = EarlyFusionBaseline()
path = train_model(ef, 'ef_baseline', lossw=0.0)
ef.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
ef.to(DEVICE)
results['Early Fusion (baseline)'] = evaluate(ef)
trained_models['Early Fusion (baseline)'] = ef
print('  ->', results['Early Fusion (baseline)'])

In [ ]:
# ── PLGModel v1: per-layer gated residual, local gates ────────────────────────
v1 = PLGModelV1(in_dims=IN_DIMS, d_model=64, n_heads=4, n_layers=NUM_LAYERS)
path = train_model(v1, 'plg_v1_lam001', lossw=0.01)
v1.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
v1.to(DEVICE)
results['PLG v1  (soft residual, λ=0.01)'] = evaluate(v1)
trained_models['PLG v1  (soft residual, λ=0.01)'] = v1
print('  ->', results['PLG v1  (soft residual, λ=0.01)'])

In [ ]:
# ── PLGModelV2: centralized gates, multiplicative zeroing ─────────────────────
v2_configs = [
    dict(name='plg_v2_soft_lam000', hard_gate=False, lossw=0.00, label='PLG v2  (soft, λ=0)'),
    dict(name='plg_v2_soft_lam001', hard_gate=False, lossw=0.01, label='PLG v2  (soft, λ=0.01)'),
    dict(name='plg_v2_hard_lam001', hard_gate=True,  lossw=0.01, label='PLG v2  (hard, λ=0.01)'),
    dict(name='plg_v2_hard_lam005', hard_gate=True,  lossw=0.05, label='PLG v2  (hard, λ=0.05)'),
]

for cfg in v2_configs:
    m = make_v2(hard_gate=cfg['hard_gate'])
    path = train_model(m, cfg['name'], lossw=cfg['lossw'])
    m.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
    m.to(DEVICE)
    results[cfg['label']] = evaluate(m)
    trained_models[cfg['label']] = m
    print(f"  -> {results[cfg['label']]}")

## 5a · DynMM-style Baseline (trained in-notebook)

**`DynMMBaseline`** implements the modality-level DynMM design from Table 2 of the paper
directly in this notebook, using the same `GatedTransformerEncoder` backbone as every
other model here. This means it shares `evaluate()` and the efficiency plot with
no special data-loader changes needed.

Key differences from `PLGModelV2`:

| Aspect | PLGModelV2 | DynMMBaseline |
|---|---|---|
| Gate granularity | Per-layer (15 decisions per sample) | Per-sample (1 of 2 experts) |
| What gets skipped | Individual transformer layers | Entire modality encoders |
| Analogous DynMM level | Fusion-level | Modality-level |

We train three variants:
- **End-to-end** (soft gate, λ=0): all weights trained from scratch, no resource pressure
- **End-to-end** (hard gate, λ=0.05): discrete branch selection with resource penalty
- **Two-stage** (hard gate, λ=0.05): E1/E2 initialized from pre-trained baselines,
  then fine-tuned with gating (mirrors DynMM Stage I → Stage II)

In [ ]:
dynmm_configs = [
    dict(name='dynmm_soft_lam000', hard_gate=False, lossw=0.00,
         pretrain=False, label='DynMM-style (soft, λ=0)'),
    dict(name='dynmm_hard_lam005', hard_gate=True,  lossw=0.05,
         pretrain=False, label='DynMM-style (hard, λ=0.05)'),
    dict(name='dynmm_hard_lam005_pretrain', hard_gate=True, lossw=0.05,
         pretrain=True,  label='DynMM-style (hard, λ=0.05, 2-stage)'),
]

for cfg in dynmm_configs:
    m = DynMMBaseline(hard_gate=cfg['hard_gate'])
    if cfg['pretrain']:
        # Stage I: seed E1/E2 from already-trained baselines
        m.init_from_baselines(
            trained_models['Text-only (baseline)'],
            trained_models['Late Fusion (baseline)']
        )
        print(f"  [{cfg['label']}] E1/E2 initialized from pre-trained baselines")
    # Stage II: fine-tune end-to-end with gating + resource loss
    path = train_model(m, cfg['name'], lossw=cfg['lossw'])
    m.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
    m.to(DEVICE)
    results[cfg['label']]       = evaluate(m)
    trained_models[cfg['label']] = m
    print(f"  -> {results[cfg['label']]}")

## 5b · DynMM Results (paste from external training)

DynMM (`affect_dyn.py`) uses packed sequences with per-modality padding masks and
cannot share `evaluate()` with this notebook. Train it via the CLI scripts below,
then paste the printed metrics into the cell below. The rest of the notebook
(cells 6 and 9) will include them automatically.

```bash
# Run from the MultiBench repo root (GPU required for full training)

# Step 1: train text-only E1 expert
python examples/affect/affect_uni.py --mod 2 --enc transformer --data mosei
cp ./log/mosei/reg_transformer_encoder_text.pt ./log/mosei/b1_reg_transformer_encoder_text.pt
cp ./log/mosei/reg_transformer_head_text.pt    ./log/mosei/b1_reg_transformer_head_text.pt

# Step 2: train late-fusion E2 expert
python examples/affect/affect_mm.py --fusion 3 --data mosei
cp ./log/mosei/lf_tran.pt ./log/mosei/b2_lf_tran.pt

# Step 3: train DynMM at different lambda values
python examples/affect/affect_dyn.py --data mosei --enc transformer --reg 0.0
python examples/affect/affect_dyn.py --data mosei --enc transformer --reg 0.05 --hard-gate
python examples/affect/affect_dyn.py --data mosei --enc transformer --reg 0.10 --hard-gate
```

Each run prints at the end:
```
Test Accuracy  XX.XX +/- ...
Loss           X.XXXX +/- ...
Corr           X.XXXX +/- ...
FLOP           XXX.XX +/- ...
```
Paste those numbers (Acc/100, MAE=Loss, Corr, FLOP) into the dict below.

In [ ]:
# ── Paste DynMM results here after running affect_dyn.py ─────────────────────
# Format: label -> {'MAE': float, 'Corr': float, 'Acc': float}
# Leave empty ({}) if you have not trained DynMM yet.
dynmm_results = {
    # 'DynMM (soft, reg=0.0)':   {'MAE': 0.0000, 'Corr': 0.0000, 'Acc': 0.0000, 'MAdds': 0.0},
    # 'DynMM (hard, reg=0.05)':  {'MAE': 0.0000, 'Corr': 0.0000, 'Acc': 0.0000, 'MAdds': 0.0},
    # 'DynMM (hard, reg=0.10)':  {'MAE': 0.0000, 'Corr': 0.0000, 'Acc': 0.0000, 'MAdds': 0.0},
}

# Merge DynMM entries into the main results dicts
for label, r in dynmm_results.items():
    results[label] = {k: v for k, v in r.items() if k != 'MAdds'}
    trained_models[label] = None  # no PyTorch model object
    if 'MAdds' in r:
        flop_data[label] = (r['MAdds'], r['Acc'] * 100)

print(f'DynMM entries added: {list(dynmm_results.keys()) or "(none yet)"}')

## 6 · Results

In [ ]:
# -- Trained models -------------------------------------------------------
print(f"{'Model':<40} {'MAE':>7} {'Corr':>7} {'Acc %':>8}")
print('-' * 66)
for name, r in results.items():
    print(f"{name:<40} {r['MAE']:>7.4f} {r['Corr']:>7.4f} {r['Acc']*100:>8.2f}")

# -- DynMM paper reference: full Table 2 (CMU-MOSEI) ----------------------
# The paper uses a GRU-based transformer; our encoder is Transformer-only,
# so absolute numbers differ but the ordering of methods should hold.
print()
print('-- DynMM paper reference (Table 2, CMU-MOSEI, V+A+T) ---------------')
print(f"{'Model':<40} {'MAE':>7} {'Acc2 %':>8} {'MAdds (M)':>11}")
print('-' * 70)
paper_table2 = [
    # (name,                            MAE,   Acc2,   MAdds)
    ('Video Network (paper)',            0.80,  69.02,  123.1),
    ('Audio Network (paper)',            0.82,  67.68,  123.3),
    ('Text Network E1 (paper)',          0.62,  78.35,  124.7),
    ('Early Fusion (paper)',             0.65,  78.45,  313.5),
    ('Late Fusion E2 (paper)',           0.60,  79.54,  309.6),
    ('DynMM-a (paper)',                  0.62,  79.07,  165.5),
    ('DynMM-b (paper)',                  0.61,  79.73,  254.5),
    ('DynMM-c (paper)',                  0.60,  79.75,  295.8),
]
for name, mae, acc, madds in paper_table2:
    print(f"{name:<40} {mae:>7.2f} {acc:>8.2f} {madds:>11.1f}")

print()
print('Acc2 = binary accuracy (positive/negative sentiment); same metric as Acc above.')
print('MAdds measured per video-audio-text tuple using the DynMM paper architecture.')

## 7 · Gate Pattern Visualization

The centralized `PerLayerGate` outputs a `[B, 3, 5]` matrix per sample.
We visualize the mean over the test set to understand which modality-layer pairs
the network learns to activate.

- **Green (g≈1)**: modality actively transforms at this layer
- **Red (g≈0)**: layer output suppressed — modality skipped at this depth

We also split by sentiment polarity to see whether gating differs between
positive and negative samples.

In [ ]:
# Use hard-gate v2 model (λ=0.01) for visualization
viz_label = 'PLG v2  (hard, λ=0.01)'
viz_model = trained_models[viz_label]
viz_model.eval()
viz_model.infer_mode = 0

viz_model.reset_weight()
all_labels = []
with torch.no_grad():
    for batch in testdata:
        v, a, t, y = batch
        viz_model([v.float().to(DEVICE), a.float().to(DEVICE), t.float().to(DEVICE)])
        all_labels.extend(y.squeeze(-1).numpy().tolist())
viz_model._tracking = False

gate_data = torch.cat(viz_model._gate_accum, dim=0).numpy()  # [N, 3, 5]
labels_arr = np.array(all_labels)
N, n_mod, n_lay = gate_data.shape

# Count unique routing patterns
binary = (gate_data > 0.5).astype(int)
patterns = [''.join(map(str, g.flatten())) for g in binary]
n_unique = len(set(patterns))

print(f'Test samples : {N:,}')
print(f'Gate shape   : {gate_data.shape}  [samples, modalities, layers]')
print(f'Unique binary patterns: {n_unique}  (max possible: {2**(n_mod*n_lay):,})')

In [ ]:
MODALITY_NAMES = ['Visual (35d)', 'Audio (74d)', 'Text (300d)']
pos_idx = np.where(labels_arr >= 0)[0]
neg_idx = np.where(labels_arr < 0)[0]

subsets = [
    (np.arange(N),  f'All samples (n={N})'),
    (pos_idx,        f'Positive sentiment (n={len(pos_idx)})'),
    (neg_idx,        f'Negative sentiment (n={len(neg_idx)})'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, (idx, title) in zip(axes, subsets):
    g = gate_data[idx].mean(axis=0)  # [3, 5]
    im = ax.imshow(g, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    for m in range(n_mod):
        for l in range(n_lay):
            val = g[m, l]
            c = 'black' if 0.2 < val < 0.8 else 'white'
            ax.text(l, m, f'{val:.2f}', ha='center', va='center',
                    fontsize=9, color=c, fontweight='bold')
    ax.set_xticks(range(n_lay))
    ax.set_xticklabels([f'L{l+1}' for l in range(n_lay)])
    ax.set_yticks(range(n_mod))
    ax.set_yticklabels(MODALITY_NAMES)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.04)

fig.suptitle(f'{viz_label} — mean gate activation over test set\nGreen=active (g≈1)   Red=suppressed (g≈0)', y=1.05)
plt.tight_layout()
out = os.path.join(CHECKPOINT_DIR, 'plg_v2_gate_heatmap.png')
plt.savefig(out, bbox_inches='tight')
plt.show()
print('Saved', out)

### 7b · Routing Pattern Distribution

Which distinct routing patterns does the model actually use?  
A pattern is a binary vector of the 15 gate decisions for one sample.

In [ ]:
counts = Counter(patterns)
print(f'Top routing patterns  (V=visual layers on, A=audio, T=text):')
print(f'  {'Pattern':<24} {'Count':>7}  {'%':>6}  Notes')
print('─' * 55)
for pat, cnt in counts.most_common(10):
    p = np.array(list(pat), dtype=int).reshape(n_mod, n_lay)
    v_on, a_on, t_on = p[0].sum(), p[1].sum(), p[2].sum()
    pct = cnt / N * 100
    note = ''
    if v_on == 0 and a_on == 0:
        note = '← text-only'
    elif v_on == n_lay and a_on == n_lay and t_on == n_lay:
        note = '← all-on'
    print(f'  V={v_on}/5  A={a_on}/5  T={t_on}/5  {cnt:>7,}  {pct:>5.1f}%  {note}')

## 8 · Infer Mode Comparison

Set `model.infer_mode` to override the learned routing at eval time:
- `0` (default) — use the learned gate network
- `1` — force text-only: only last modality active, all others zeroed
- `-1` — force all-on: all gates = 1 (equivalent to ungated late fusion)

This reveals the accuracy cost of ablating each modality at inference without retraining.

In [ ]:
infer_cfg = {0: 'Learned (mode=0)', 1: 'Text-only (mode=1)', -1: 'All-on (mode=-1)'}
infer_results = {}

for mode, label in infer_cfg.items():
    viz_model.infer_mode = mode
    r = evaluate(viz_model)
    infer_results[label] = r
    print(f'{label:<28}: MAE={r["MAE"]:.4f}  Corr={r["Corr"]:.4f}  Acc={r["Acc"]*100:.2f}%')

viz_model.infer_mode = 0  # restore

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
names  = list(infer_results.keys())
accs   = [infer_results[n]['Acc'] * 100 for n in names]
colors = ['steelblue', 'darkorange', 'seagreen']
bars   = ax.bar(names, accs, color=colors, edgecolor='white', linewidth=1.5)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold')
lf_acc = results['Late Fusion (baseline)']['Acc'] * 100
ax.axhline(lf_acc, color='gray', linestyle='--', linewidth=1.2, label=f'LF baseline ({lf_acc:.2f}%)')
ax.set_ylabel('Test Accuracy (pos/neg %)')
ax.set_title(f'{viz_label}\nAccuracy under different infer modes')
ax.set_ylim(min(accs) - 3, max(accs) + 3)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9 · Accuracy vs. Efficiency

`cal_flop()` estimates effective MAdds as:
```
total = base_flop (135M) + Σ(mean_gates_over_test × flop_cost_per_layer)
flop_cost[m, l] = d_models[m]² × 4 / 1e6
```

With hard gates, closed gates contribute **exactly zero** to this sum.
Higher λ encourages more closed gates → lower effective FLOPs.

> Note: Paper reference MAdds use a GRU-based DynMM architecture (different dims),
> so the x-axis scales are not directly comparable — the comparison illustrates
> the accuracy-efficiency tradeoff within each method family.

In [ ]:
flop_data = {}   # label -> (flops, acc)

for label, m in trained_models.items():
    if m is None or not hasattr(m, 'cal_flop'):
        continue
    m.reset_weight()
    m.eval()
    with torch.no_grad():
        for batch in testdata:
            v, a, t, _ = batch
            m([v.float().to(DEVICE), a.float().to(DEVICE), t.float().to(DEVICE)])
    # PLGModelV2 uses _tracking; DynMMBaseline accumulates _gate_log automatically
    if hasattr(m, '_tracking'):
        m._tracking = False
    f = m.cal_flop()
    flop_data[label] = (f, results[label]['Acc'] * 100)
    print(f'{label}: {f:.3f}M  acc={results[label]["Acc"]*100:.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Paper reference points from DynMM Table 2 (different architecture/scale)
paper_pts = [
    # (label,                     MAdds,  Acc2,   marker, color)
    ('Video-only E (paper)',       123.1,  69.02, '^', 'green'),
    ('Audio-only E (paper)',       123.3,  67.68, '^', 'brown'),
    ('Text-only E1 (paper)',       124.7,  78.35, '^', 'gray'),
    ('Early Fusion (paper)',       313.5,  78.45, '^', 'cyan'),
    ('Late Fusion E2 (paper)',     309.6,  79.54, '^', 'blue'),
    ('DynMM-a (paper)',            165.5,  79.07, 'D', 'orange'),
    ('DynMM-b (paper)',            254.5,  79.73, 'D', 'orange'),
    ('DynMM-c (paper)',            295.8,  79.75, 'D', 'orange'),
]
dynmm_legend_added = False
for label, madds, acc, mk, col in paper_pts:
    leg = label
    if 'DynMM' in label:
        leg = 'DynMM variants (paper)' if not dynmm_legend_added else '_nolegend_'
        dynmm_legend_added = True
    ax.scatter(madds, acc, marker=mk, s=90, c=col, zorder=5, label=leg)
    short = label.replace(' (paper)', '')
    ax.annotate(short, (madds, acc), textcoords='offset points',
                xytext=(4, 3), fontsize=7, color='dimgray')

# Trained baselines (approximate MAdds for our smaller architecture)
baseline_pts = [
    ('Video-only (baseline)',   120.0, 's', 'green'),
    ('Audio-only (baseline)',   120.0, 's', 'brown'),
    ('Text-only (baseline)',    120.0, 's', 'gray'),
    ('Early Fusion (baseline)', 120.0, 's', 'cyan'),
    ('Late Fusion (baseline)',  120.0, 's', 'blue'),
]
for label, madds, mk, col in baseline_pts:
    if label in results:
        ax.scatter(madds, results[label]['Acc'] * 100,
                   marker=mk, s=90, c=col, zorder=5, label=f'{label} (ours)')

# PLG v1
v1_flops = 135.0 + NUM_LAYERS * (64 ** 2) * 4 / 1e6 * 3  # rough estimate
ax.scatter(
    v1_flops, results['PLG v1  (soft residual, λ=0.01)']['Acc'] * 100,
    marker='P', s=100, c='purple', zorder=5, label='PLG v1 (soft residual)'
)

# PLG v2 Pareto curve
v2_labels = sorted(flop_data.keys(), key=lambda k: flop_data[k][0])
v2_f   = [flop_data[l][0] for l in v2_labels]
v2_acc = [flop_data[l][1] for l in v2_labels]
ax.plot(v2_f, v2_acc, 'o-', color='red', markersize=9, linewidth=2,
        label='PLG v2 (ours)', zorder=6)
for l, f, acc in zip(v2_labels, v2_f, v2_acc):
    short = l.replace('PLG v2  ', '')
    ax.annotate(short, (f, acc), textcoords='offset points',
                xytext=(5, 4), fontsize=8)

ax.set_xlabel('Effective MAdds (M)  [paper and ours use different architectures]', fontsize=10)
ax.set_ylabel('Test Accuracy % (pos/neg)', fontsize=11)
ax.set_title(
    'CMU-MOSEI: Accuracy vs. Computation\n'
    '(triangles/diamonds = DynMM paper baselines; squares = our trained baselines)',
    fontsize=11)
ax.legend(fontsize=7, loc='lower right', ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = os.path.join(CHECKPOINT_DIR, 'plg_v2_acc_vs_efficiency.png')
plt.savefig(out, bbox_inches='tight')
plt.show()
print('Saved', out)

## Summary

### What changed in v2

| Aspect | v1 (`per_layer_gated.py`) | v2 (`per_layer_gated_v2.py`) |
|---|---|---|
| Gate source | Local: `Linear(d_model→1)` per `(m,l)`, from that modality's own hidden state | **Centralized**: one `PerLayerGate` sees all raw inputs, outputs full `[B,M,L]` upfront |
| Gate mechanism | Gated residual `g·h_new + (1-g)·h_prev` | **Multiplicative zeroing** `h = h * g` |
| Hard gating | ✗ | **✓** straight-through sigmoid |
| Resource loss | `gates.mean()` (uniform) | **FLOPs-weighted** `Σ(g × cost)` |
| Infer modes | ✗ | **✓** 0=learned · 1=text-only · -1=all-on |
| Pretrained init | ✗ | **✓** `init_from_pretrained()` |

### Key files

| File | Role |
|---|---|
| `fusions/per_layer_gated_v2.py` | v2 model — centralized gates, hard gating, FLOPs loss |
| `fusions/per_layer_gated.py` | v1 model — local gated residuals |
| `examples/affect/affect_plg.py` | CLI training script (v1) |
| `examples/affect/demo_per_layer_gates_clean.ipynb` | v1 demo notebook |
| `examples/affect/demo_per_layer_gates_v2.ipynb` | **this notebook** |